# Acoustic-Aware Fusion — severity LOSO training (local, RTX 4060)

End-to-end, resumable pipeline for the three-branch severity model on UA-Speech M6:

1. **Environment** — locate the project, report the hardware, apply the local runtime profile.
2. **Data** — use the extracted corpus under `data/extracted` (extract `data/raw/*.tgz` only if it is missing), load the manifest.
3. **Feature store** — build only the missing chunks (resumable), verify 300 VAD spans and 24 feature tensors bit-exact against the live pipeline.
4. **Sanity checks** — one real batch, the model's parameter audit.
5. **Throughput** — recorded RTX 4060 numbers (or an optional fresh benchmark) and the projected run time.
6. **Training** — full 15-fold LOSO; re-running this notebook resumes finished folds from disk and an interrupted fold from its last epoch.
7. **Results** — per-fold table, pooled evaluation, confusion matrix.

Every path comes from `src/config.py` (repo-relative). Hardware settings live in its *Local hardware profile* block.

In [ ]:
# -----------------------------------------------------------------------------
# 1.1  Imports and project root
# -----------------------------------------------------------------------------
import time
SESSION_START = time.monotonic()

import json
import os
import platform
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# The notebook may be started from notebooks/ or from the repo root: walk up
# until the directory holding src/config.py is found.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src import feature_store
from src.console import (print_banner, print_header, print_kv, print_note, print_status,
                         print_table, silence_library_progress)
from src.training.data import load_manifest
from src.training.models import SEVERITY_MODEL_NAME, build_model, parameter_counts
from src.training.reporting import (check_frozen_config_guard, print_feature_audit,
                                    print_final_run_configuration, summarize_registry,
                                    write_frozen_config)
from src.training.runner import TrainingConfig, run_training
from src.training.session import ThroughputMeasurement, project_runtime, subset_blocks
from src.training.utils import configure_local_runtime, resolve_amp_dtype, resolve_device
from src.vad_cache import verify_vad_span_cache

config.ensure_directories()
silence_library_progress()     # no per-parameter "Loading weights" bars on every model build
print_status(f"Project root: {PROJECT_ROOT}", ok=True)

In [ ]:
# -----------------------------------------------------------------------------
# 1.2  Hardware and local runtime profile
# -----------------------------------------------------------------------------
import psutil

device = resolve_device(None)
if device.type != "cuda":
    raise RuntimeError("CUDA is not available to this kernel — select the Python environment "
                       "with the CUDA build of torch (e.g. the 'torch-gpu' conda env).")
configure_local_runtime(device)          # cudnn.benchmark + VRAM cap (config.CUDA_MEMORY_FRACTION)

props = torch.cuda.get_device_properties(device)
ram = psutil.virtual_memory()
disk = shutil.disk_usage(PROJECT_ROOT)

print_header("Environment")
print_kv("Python / platform", f"{platform.python_version()} / {platform.platform()}")
print_kv("PyTorch / CUDA", f"{torch.__version__} / {torch.version.cuda}")
print_kv("GPU", f"{props.name}, {props.total_memory / 2**30:.1f} GiB, sm_{props.major}{props.minor}")
print_kv("CPU", f"{psutil.cpu_count(logical=False)} cores / {psutil.cpu_count()} threads")
print_kv("RAM", f"{ram.total / 2**30:.1f} GiB total, {ram.available / 2**30:.1f} GiB available")
print_kv("Disk free (project drive)", f"{disk.free / 2**30:.0f} GiB")

print_header("Local runtime profile (src/config.py)")
print_kv("AMP dtype", str(resolve_amp_dtype(device)).replace("torch.", "")
         + (" + GradScaler" if resolve_amp_dtype(device) == torch.float16 else ""))
print_kv("Batch size", config.DEFAULT_BATCH_SIZE)
print_kv("wav2vec2 gradient checkpointing", config.WAV2VEC_GRADIENT_CHECKPOINTING)
print_kv("DataLoader workers (train / eval)", f"{config.TRAIN_NUM_WORKERS} / {config.EVAL_NUM_WORKERS}"
         f", prefetch {config.DATALOADER_PREFETCH_FACTOR}, pinned memory")
print_kv("VRAM cap", f"{config.CUDA_MEMORY_FRACTION:.0%} of {props.total_memory / 2**30:.1f} GiB")
print_kv("cuDNN autotune", torch.backends.cudnn.benchmark)
print_kv("Feature-store build workers", config.FEATURE_STORE_WORKERS)
if ram.available < 6 * 2**30:
    print_note("Less than 6 GiB of RAM is free — close other applications before training "
               "(training uses ~5-7 GiB across the main process and DataLoader workers).")

In [ ]:
# -----------------------------------------------------------------------------
# 2.1  Corpus — extracted audio under data/extracted, else extract data/raw/*.tgz
# -----------------------------------------------------------------------------
from src.extraction import extract_all_archives
from src.scanning import scan_audio_files, verify_speakers

EXPECTED_M6 = len(config.ALL_SPEAKERS) * config.WORDS_PER_SPEAKER          # 28 x 765
present_m6 = (sum(1 for _ in config.AUDIO_DIR.rglob(f"*_{config.TARGET_MIC}.wav"))
              if config.AUDIO_DIR.exists() else 0)
print_kv("Audio folder", config.AUDIO_DIR)
if present_m6 >= EXPECTED_M6:
    print_status(f"{present_m6:,} {config.TARGET_MIC} files present — no extraction needed.", ok=True)
else:
    archives = [config.ARCHIVE_DIR / name for name in config.ARCHIVE_FILES]
    missing = [str(a) for a in archives if not a.exists()]
    if missing:
        raise FileNotFoundError(f"Only {present_m6:,}/{EXPECTED_M6:,} M6 files extracted and the "
                                f"archives are missing: {missing}")
    print_kv("Extracting", f"{present_m6:,}/{EXPECTED_M6:,} present — extracting from {config.ARCHIVE_DIR}")
    extract_all_archives()
    verify_speakers(scan_audio_files())

In [ ]:
# -----------------------------------------------------------------------------
# 2.2  Manifest
# -----------------------------------------------------------------------------
df_full = load_manifest()          # outputs/m6_manifest.csv (regenerated from audio if absent)
missing_files = int((~df_full["Filepath"].map(os.path.exists)).sum())
if missing_files:
    raise FileNotFoundError(f"{missing_files:,} manifest files do not exist on disk — "
                            f"delete {config.MANIFEST_PATH} to regenerate it from {config.AUDIO_DIR}.")

print_header("Manifest")
print_kv("Utterances (M6)", f"{len(df_full):,}")
print_kv("Speakers", df_full["Speaker_ID"].nunique())
composition = (df_full.groupby(["Severity", "Block"]).size().unstack(fill_value=0)
               .rename_axis(index="Severity", columns=None).reset_index())
composition["total"] = composition[["B1", "B2", "B3"]].sum(axis=1)
print_table(composition)

In [ ]:
# -----------------------------------------------------------------------------
# 2.3  Run parameters
# -----------------------------------------------------------------------------
RUN_BLOCKS = ["B1"]                     # the B1 severity experiment (255 utterances/speaker)
MODEL = SEVERITY_MODEL_NAME             # or an ablation: ab1_wav2vec2_only, ab2_acoustic_only,
                                        # ab3_wav2vec2_acoustic_concat, ab4_wav2vec2_segmental,
                                        # ab5_wav2vec2_suprasegmental, ab6_full_fusion,
                                        # ab7_full_complementarity, ab8_full_speaker_grl
EPOCHS, PATIENCE = 12, 3
BATCH_SIZE = config.DEFAULT_BATCH_SIZE                         # 32 (RTX 4060 benchmark)
GRADIENT_CHECKPOINTING = config.WAV2VEC_GRADIENT_CHECKPOINTING  # False (RTX 4060 benchmark)
CACHE_SCOPE = "run"                     # "run": the 15 dysarthric speakers x RUN_BLOCKS; "all": 21,420 files
RUN_BENCHMARK = False                   # True: re-measure batch size x checkpointing (~7 min)

# SMOKE_TEST (or env AAFA_SMOKE=1): 2 folds, 1 epoch, 64 utterances per split,
# under a separate run name — an end-to-end check in ~2 minutes.
SMOKE_TEST = os.environ.get("AAFA_SMOKE") == "1"

RUN_NAME = f"sev_{MODEL}_{'+'.join(RUN_BLOCKS)}_spkval_v2" + ("_smoke" if SMOKE_TEST else "")
df_run = subset_blocks(df_full, RUN_BLOCKS)
print_kv("Run name", RUN_NAME)
print_kv("Utterances in run", f"{len(df_run):,} ({'+'.join(RUN_BLOCKS)})")
if SMOKE_TEST:
    print_note("SMOKE TEST — pipeline check only, not a result.")

In [ ]:
# -----------------------------------------------------------------------------
# 3.1  Feature store — build only the missing chunks (resumable)
# -----------------------------------------------------------------------------
# One compressed chunk per (speaker, block) under outputs/feature_cache/store,
# each written atomically as it finishes: interrupting this cell loses at most
# the chunks in flight, and re-running it builds only what is missing.
if CACHE_SCOPE == "all":
    df_cache = df_full
else:
    df_cache = df_run[df_run["Speaker_ID"].isin(config.DYSARTHRIC_IDS)].reset_index(drop=True)

feature_store.build_feature_store(df_cache, n_workers=config.FEATURE_STORE_WORKERS)
coverage = feature_store.feature_store_coverage(df_cache)
print_kv("Feature-store coverage", f"{coverage['files_done']:,} / {coverage['files_total']:,} utterances, "
                                   f"{coverage['chunks_done']} / {coverage['chunks_total']} chunks")
if not coverage["complete"]:
    raise RuntimeError(f"Feature store incomplete (missing {coverage['chunks_missing']}) — re-run this cell.")

In [ ]:
# -----------------------------------------------------------------------------
# 3.2  Verification — 300 VAD spans vs live Silero, 24 feature tensors bit-exact
# -----------------------------------------------------------------------------
verify_vad_span_cache(df_cache, n=300)
feature_store.verify_feature_store(df_cache, n=24)

In [ ]:
# -----------------------------------------------------------------------------
# 4.1  One real batch, end to end
# -----------------------------------------------------------------------------
from torch.utils.data import DataLoader
from src.dataset import UASpeechDataset

sample_df = df_cache.sample(n=8, random_state=config.DEFAULT_SEED).reset_index(drop=True)
batch = next(iter(DataLoader(UASpeechDataset(sample_df, include_three_branch=True),
                             batch_size=4, shuffle=False)))
print_header("One batch")
for key in ("waveform", "segmental", "supra", "waveform_length", "supra_valid_frames", "severity_label"):
    print_kv(key, f"{tuple(batch[key].shape)}  {batch[key].dtype}")
assert batch["segmental"].shape[1] == config.SEGMENTAL_CHANNELS
assert batch["supra"].shape[1] == config.SUPRA_CHANNELS
assert batch["waveform"].shape[-1] == config.MAX_SAMPLES
print_status("Batch shapes match the architecture contract.", ok=True)

In [ ]:
# -----------------------------------------------------------------------------
# 4.2  Architecture and parameter counts
# -----------------------------------------------------------------------------
model_preview = build_model(MODEL, config.NUM_CLASSES["severity"], num_speakers=10,
                            gradient_checkpointing=GRADIENT_CHECKPOINTING)
print_header(f"Model — {MODEL}")
for name, value in parameter_counts(model_preview).items():
    print_kv(name, f"{value:,.2f}" if isinstance(value, float) else f"{value:,}")
if MODEL == SEVERITY_MODEL_NAME:
    print_feature_audit(model_preview, num_classes=config.NUM_CLASSES["severity"])
del model_preview
torch.cuda.empty_cache()

In [ ]:
# -----------------------------------------------------------------------------
# 5.1  Throughput and projected run time
# -----------------------------------------------------------------------------
if RUN_BENCHMARK:
    from src.training.session import benchmark_batch_sizes
    measurement, benchmark_table = benchmark_batch_sizes(
        df_run, model_name=MODEL, task="severity", batch_sizes=(32, 64))
    BATCH_SIZE, GRADIENT_CHECKPOINTING = measurement.batch_size, measurement.gradient_checkpointing
else:
    # Recorded on this laptop (RTX 4060 Laptop, batch 32, checkpointing off).
    measurement = ThroughputMeasurement(train_samples_per_s=76.3, eval_samples_per_s=170.7,
                                        batch_size=32, device="cuda", amp_dtype="float16",
                                        n_train_measured=192, n_eval_measured=64,
                                        peak_memory_mb=5828, gradient_checkpointing=False)

projection = project_runtime(df_run, measurement, task="severity", model_name=MODEL, epochs=EPOCHS)
print_header("Projected run time")
print_kv("Batch size / grad. checkpointing", f"{BATCH_SIZE} / {GRADIENT_CHECKPOINTING}")
print_kv("Throughput (train / eval)", f"{measurement.train_samples_per_s:.1f} / "
                                      f"{measurement.eval_samples_per_s:.1f} samples/s")
print_kv("Mean fold size (train/val/test)", f"{projection['n_train']} / {projection['n_val']} / {projection['n_test']}")
print_kv("Epoch", f"{projection['epoch_seconds'] / 60:.1f} min")
print_kv("Fold at max epochs", f"{projection['fold_seconds'] / 60:.1f} min")
print_kv(f"{projection['n_folds']} folds, upper bound", f"{projection['total_seconds'] / 3600:.1f} h "
                                                        f"(early stopping only shortens it)")

In [ ]:
# -----------------------------------------------------------------------------
# 6.1  Frozen configuration
# -----------------------------------------------------------------------------
cfg = TrainingConfig(
    task="severity",
    model=MODEL,
    run_name=RUN_NAME,
    epochs=1 if SMOKE_TEST else EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    num_workers=config.TRAIN_NUM_WORKERS,
    eval_num_workers=config.EVAL_NUM_WORKERS,
    seed=config.DEFAULT_SEED,
    val_protocol="speaker",
    max_folds=2 if SMOKE_TEST else None,
    limit_samples=64 if SMOKE_TEST else None,
)
final_config = print_final_run_configuration(cfg, df_run)
check_frozen_config_guard(final_config)      # refuse a changed config under the same RUN_NAME...
write_frozen_config(final_config)            # ...before recording this one

In [ ]:
# -----------------------------------------------------------------------------
# 6.2  Training — 15-fold LOSO, resumable
# -----------------------------------------------------------------------------
# Finished folds (metrics + predictions on disk) are loaded, not retrained; a
# fold interrupted mid-way resumes from checkpoints/<RUN_NAME>/<fold>/latest.pt.
# To resume after closing the notebook or a crash: just run the notebook again.
training_start = time.monotonic()
summary, pooled_metrics = run_training(df_run, cfg)
print()
print_kv("Training wall-clock", f"{(time.monotonic() - training_start) / 3600:.2f} h")
print_kv("Peak GPU memory", f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

In [ ]:
# -----------------------------------------------------------------------------
# 7.1  Per-fold results and coverage
# -----------------------------------------------------------------------------
run_status = json.loads((config.METRICS_DIR / RUN_NAME / "RUN_STATUS.json").read_text())
print_banner(f"SEVERITY — {run_status['status']}",
             f"{run_status['completed']}/{run_status['expected']} folds, {'+'.join(RUN_BLOCKS)}")
if run_status["status"] != "COMPLETE":
    print_note(f"Missing folds {run_status['missing_folds']} — not a final result; re-run to resume.")
per_fold = pd.read_csv(config.METRICS_DIR / f"{RUN_NAME}.per_fold.csv")
columns = [c for c in ("fold", "true_label", "n_samples", "accuracy", "ordinal_mae", "best_epoch",
                       "epochs_completed", "train_time_s", "val_speakers") if c in per_fold]
print_table(per_fold[columns].round(4))
print_note("Per-fold macro-F1 / balanced accuracy / AUROC: N/A — each held-out speaker has one true class.")
print()
print_header("Pooled across completed folds")
for name, value in pooled_metrics.items():
    print_kv(name, f"{value:.4f}" if isinstance(value, float) else value)

In [ ]:
# -----------------------------------------------------------------------------
# 7.2  Confusion matrix (pooled over completed folds)
# -----------------------------------------------------------------------------
predictions = pd.concat([pd.read_csv(p) for p in sorted((config.PREDICTIONS_DIR / RUN_NAME).glob("*.csv"))],
                        ignore_index=True)
class_names = config.SEVERITY_CLASS_NAMES
matrix = (pd.crosstab(predictions["y_true"], predictions["y_pred"])
          .reindex(index=range(4), columns=range(4), fill_value=0).to_numpy())
figure, axis = plt.subplots(figsize=(5.5, 4.6))
image = axis.imshow(matrix, cmap="Blues")
axis.set_xticks(range(4), class_names, rotation=30, ha="right")
axis.set_yticks(range(4), class_names)
axis.set_xlabel("Predicted severity")
axis.set_ylabel("True severity")
axis.set_title(f"{len(predictions):,} held-out utterances — {run_status['status']}")
for i in range(4):
    for j in range(4):
        axis.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center",
                  color="white" if matrix[i, j] > matrix.max() / 2 else "black")
figure.colorbar(image, ax=axis, shrink=0.8)
figure.tight_layout()
plt.show()
print_kv("Artifacts", config.OUTPUT_DIR)
print_kv("Whole notebook", f"{(time.monotonic() - SESSION_START) / 3600:.2f} h")

## Resuming and ablations

* **Resume**: re-run the notebook (Run All). The feature store skips finished chunks; training skips finished folds and resumes an interrupted fold from its last saved epoch.
* **Ablations**: set `MODEL` in 2.3 to one of the `ab*` names and Run All — same data, folds, validation and optimizer; each gets its own `RUN_NAME`. `ab2_acoustic_only` has no wav2vec2 and runs in minutes.
* **Start a configuration over**: delete `outputs/{checkpoints,metrics,predictions}/<RUN_NAME>`.